# ML-TTS Audio Samples — Drive Download + Embedded Parquet

Downloads all **100 audio samples** (50 English + 50 Hindi) from Google Drive and builds an embedded Parquet file.

**Before running:** make sure your Google account can open the [ML-TTS Audio Samples folder](https://drive.google.com/drive/folders/1E-3-uCb_XpLq6nY7hhSIUvN1kMR6IWbR?usp=sharing).

Two download methods are provided:
1. **Drive API (recommended)** — works for private/shared files after Google sign-in
2. **gdown fallback** — only works if files are public ("Anyone with the link")

In [ ]:
!pip install -q openpyxl gdown pyarrow

In [ ]:
import io
import json
import re
import shutil
import zipfile
from pathlib import Path

import openpyxl
import pyarrow as pa
import pyarrow.parquet as pq
from google.colab import auth, files
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import google.auth

WORKDIR = Path('/content/ml_tts')
AUDIO_DIR = WORKDIR / 'audio'
WORKDIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_FOLDER_ID = '1E-3-uCb_XpLq6nY7hhSIUvN1kMR6IWbR'
DRIVE_ID_RE = re.compile(r'/file/d/([a-zA-Z0-9_-]+)')
AUDIO_EXTENSIONS = {'.wav', '.mp3', '.m4a', '.flac', '.ogg', '.webm', '.aac'}

## Step 1 — Upload the spreadsheet

Upload `ML-TTS Audio Samples.xlsx` from your machine (or skip if you already uploaded it to Colab).

In [ ]:
print('Upload ML-TTS Audio Samples.xlsx')
uploaded = files.upload()
xlsx_name = next(iter(uploaded))
xlsx_path = WORKDIR / xlsx_name
xlsx_path.write_bytes(uploaded[xlsx_name])
print(f'Saved -> {xlsx_path}')

In [ ]:
def extract_drive_id(url: str) -> str:
    match = DRIVE_ID_RE.search(url)
    if not match:
        raise ValueError(f'Bad Drive URL: {url}')
    return match.group(1)


def parse_sheet_name(sheet_name: str) -> tuple[str, str]:
    speaker = sheet_name.split('(')[0].strip()
    language = 'en' if '(E)' in sheet_name else 'hi'
    return speaker, language


def parse_xlsx(path: Path) -> list[dict]:
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    rows = []
    for sheet_name in wb.sheetnames:
        if sheet_name == 'Home':
            continue
        speaker, language = parse_sheet_name(sheet_name)
        ws = wb[sheet_name]
        for row in ws.iter_rows(values_only=True):
            if not row or row[0] is None or not isinstance(row[0], (int, float)):
                continue
            rows.append({
                'script_id': int(row[0]),
                'language': language,
                'speaker': speaker,
                'domain': str(row[1] or ''),
                'named_entity': str(row[2] or ''),
                'text': str(row[3] or '').strip(),
                'transliteration': str(row[4] or ''),
                'meaning': str(row[5] or '').strip(),
                'source_url': str(row[6]),
                'drive_id': extract_drive_id(str(row[6])),
            })
    wb.close()
    rows.sort(key=lambda r: (r['language'], r['script_id'], r['speaker']))
    return rows


rows = parse_xlsx(xlsx_path)
manifest_path = WORKDIR / 'samples_manifest.json'
manifest_path.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Parsed {len(rows)} samples ({sum(1 for r in rows if r["language"]=="en")} en, {sum(1 for r in rows if r["language"]=="hi")} hi)')

## Step 2 — Authenticate with Google

Sign in with the Google account that has access to the ML-TTS folder.

In [ ]:
auth.authenticate_user()
creds, _ = google.auth.default()
service = build('drive', 'v3', credentials=creds)
print('Google Drive API ready')

In [ ]:
def detect_extension(data: bytes) -> str:
    header = data[:12]
    if header[:4] == b'RIFF':
        return '.wav'
    if header[:3] == b'ID3' or header[:2] in (b'\xff\xfb', b'\xff\xf3', b'\xff\xf2'):
        return '.mp3'
    if len(header) >= 8 and header[4:8] == b'ftyp':
        return '.m4a'
    if header[:4] == b'fLaC':
        return '.flac'
    if header[:4] == b'OggS':
        return '.ogg'
    return '.bin'


def download_by_file_id(file_id: str) -> bytes:
    request = service.files().get_media(fileId=file_id)
    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    return buffer.getvalue()


def download_all_samples(rows: list[dict]) -> list[dict]:
    records = []
    failed = []

    for i, row in enumerate(rows, start=1):
        out_base = AUDIO_DIR / f"{row['language']}_{row['script_id']:02d}_{row['speaker']}"
        existing = [p for p in AUDIO_DIR.glob(f"{out_base.name}.*") if p.suffix.lower() in AUDIO_EXTENSIONS]
        if existing:
            audio_path = existing[0]
            print(f'[{i}/{len(rows)}] skip (exists) {audio_path.name}')
        else:
            try:
                data = download_by_file_id(row['drive_id'])
                ext = detect_extension(data)
                audio_path = out_base.with_suffix(ext)
                audio_path.write_bytes(data)
                print(f'[{i}/{len(rows)}] downloaded {audio_path.name} ({len(data)/1024:.1f} KB)')
            except Exception as exc:
                failed.append((row, str(exc)))
                print(f'[{i}/{len(rows)}] FAILED {row["speaker"]} #{row["script_id"]}: {exc}')
                continue

        records.append({
            **{k: row[k] for k in row if k != 'drive_id'},
            'audio_bytes': audio_path.read_bytes(),
            'audio_path': audio_path.name,
        })

    if failed:
        print(f'\n{len(failed)} downloads failed. Check folder access for your account.')
    return records

In [ ]:
records = download_all_samples(rows)
print(f'\nReady to embed: {len(records)}/{len(rows)} samples')

## Step 3 — Build embedded Parquet

In [ ]:
def write_embedded_parquet(records: list[dict], output: Path) -> None:
    audio_struct = pa.StructArray.from_arrays(
        [
            pa.array([r['audio_bytes'] for r in records], type=pa.binary()),
            pa.array([r['audio_path'] for r in records], type=pa.string()),
        ],
        names=['bytes', 'path'],
    )
    table = pa.table({
        'script_id': pa.array([r['script_id'] for r in records], type=pa.int32()),
        'language': [r['language'] for r in records],
        'speaker': [r['speaker'] for r in records],
        'domain': [r['domain'] for r in records],
        'named_entity': [r['named_entity'] for r in records],
        'text': [r['text'] for r in records],
        'transliteration': [r['transliteration'] for r in records],
        'meaning': [r['meaning'] for r in records],
        'source_url': [r['source_url'] for r in records],
        'audio': audio_struct,
    })
    pq.write_table(table, output)


parquet_path = WORKDIR / 'ml_tts_samples_embedded.parquet'
write_embedded_parquet(records, parquet_path)
print(f'Wrote {parquet_path} ({parquet_path.stat().st_size / (1024*1024):.1f} MB, {len(records)} rows)')

## Step 4 — Download results to your machine

In [ ]:
zip_path = WORKDIR / 'ml_tts_bundle.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(parquet_path, parquet_path.name)
    zf.write(manifest_path, manifest_path.name)
    for audio_file in sorted(AUDIO_DIR.iterdir()):
        if audio_file.is_file():
            zf.write(audio_file, f'audio/{audio_file.name}')

print(f'Bundle size: {zip_path.stat().st_size / (1024*1024):.1f} MB')
files.download(str(zip_path))

---
## Optional — gdown fallback (public links only)

Run this **only** if the Drive API method above fails and the files are shared as **Anyone with the link**.

In [ ]:
# !pip install -q gdown
# import gdown

# for i, row in enumerate(rows, start=1):
#     out = AUDIO_DIR / f"{row['language']}_{row['script_id']:02d}_{row['speaker']}.bin"
#     if out.exists():
#         continue
#     url = f"https://drive.google.com/uc?id={row['drive_id']}"
#     gdown.download(url, str(out), quiet=False, fuzzy=True)
#     print(f'[{i}/{len(rows)}] {out.name}')

---
## Optional — copy from mounted Drive folder

If the samples folder is already in **your** Google Drive (e.g. you added a shortcut), mount and copy directly:

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# Source folder inside *your* Drive that contains speaker subfolders/files.
# Adjust this path to wherever the ML-TTS folder appears after mounting.
# SOURCE = Path('/content/drive/MyDrive/ML-TTS Audio Samples')

# for path in SOURCE.rglob('*'):
#     if path.suffix.lower() in AUDIO_EXTENSIONS:
#         dest = AUDIO_DIR / path.name
#         if not dest.exists():
#             shutil.copy2(path, dest)
#             print('copied', path.name)